# Decision Tree Fraud Detection: Rule Extraction & Feature Importance

This notebook trains Decision Tree models on Spark to:
1. **Train interpretable Decision Tree classifiers** for fraud detection
2. **Extract human-readable rules** from the trained trees
3. **Analyze feature importance** to understand key fraud indicators
4. **Generate actionable fraud detection rules** for production systems

## Why Decision Trees for Fraud Detection?
- **Interpretability**: Easy to explain decisions to stakeholders
- **Rule Extraction**: Can convert to if-then rules for rule engines
- **No Feature Scaling**: Works with raw features
- **Handles Non-linearity**: Captures complex patterns
- **Feature Importance**: Identifies most critical fraud indicators

**Date**: November 12, 2025  
**Dataset**: JazzCash Fraud Features from ClickHouse

## 1. Setup and Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col, when, count
import pyspark.sql.functions as F
from datetime import datetime
import logging
import json
import os

# Configure logging
log_filename = f"decision_tree_training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
log_path = f"/root/research-dir/dev/jazzcash-fraud-detection/models/logs/{log_filename}"
os.makedirs(os.path.dirname(log_path), exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('decision_tree_fraud')

logger.info("✅ Libraries imported successfully")
logger.info(f"📝 Logging to: {log_path}")
print(f"✅ Setup complete! Logging to: {log_path}")

2025-11-17 14:28:18,775 - INFO - ✅ Libraries imported successfully
2025-11-17 14:28:18,776 - INFO - 📝 Logging to: /root/research-dir/dev/jazzcash-fraud-detection/models/logs/decision_tree_training_20251117_142818.log


✅ Setup complete! Logging to: /root/research-dir/dev/jazzcash-fraud-detection/models/logs/decision_tree_training_20251117_142818.log


## 2. Initialize Spark Session

In [2]:
# Configuration
jar_files = [
    "/root/research-dir/dev/jazzcash-fraud-detection/utils/clickhouse-jdbc-0.9.2-all-dependencies.jar"
]

CLICKHOUSE_CONFIG = {
    'host': 'localhost',
    'port': 9000,
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'
}

url = f"jdbc:ch://{CLICKHOUSE_CONFIG['host']}:8123/{CLICKHOUSE_CONFIG['database']}"
user = CLICKHOUSE_CONFIG['user']
password = CLICKHOUSE_CONFIG['password']
driver = "com.clickhouse.jdbc.ClickHouseDriver"

# Stop existing Spark session if present
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        logger.info("🔄 Stopped existing Spark session")
except:
    pass

# Create Spark session optimized for ML workloads
spark = SparkSession.builder \
    .appName("DecisionTreeFraudDetection") \
    .master("spark://dfs-ai-app2:7077") \
    .config("spark.jars", ",".join(jar_files)) \
    .config("spark.executor.memory", "150g") \
    .config("spark.executor.memoryOverhead", "5g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "32") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "96") \
    .getOrCreate()

logger.info("✅ Spark session initialized")
logger.info(f"   • Spark Version: {spark.version}")
logger.info(f"   • Master: {spark.sparkContext.master}")
print("✅ Spark session ready!")

25/11/17 14:28:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 14:29:25 WARN StandaloneSchedulerBackend: Application ID is not initialized yet.
25/11/17 14:29:25 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: All masters are unresponsive! Giving up.
25/11/17 14:29:25 WARN StandaloneAppClient$ClientEndpoint: Drop UnregisterApplication(null) because has not yet connected to master


Py4JJavaError: An error occurred while calling None.org.apache.spark.sql.classic.SparkSession.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:75)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:53)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:502)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:486)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(Constru

## 3. Load Fraud Detection Features from ClickHouse

In [ ]:
# Date range for training data
start_date = '2025-06-01'
end_date = '2025-06-30'
num_partitions = 30

# Selected features based on fraud profiling analysis
selected_cols = [
    'cutoff_date',
    'fraud_flag',
    'trx_channel',
    'trx_type',
    'start_balance',
    'trx_amt',
    'mbar_registered_channel',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'is_night',
    'is_business_hours',
    'is_unusual_hour',
    'night_weekend_combo',
    'txn_txns_3d',
    'txn_total_amount_3d',
    'txn_avg_amount_3d',
    'txn_max_amount_3d',
    'txn_min_amount_3d',
    'txn_unique_recipients_3d',
    'txn_unique_channels_3d',
    'txn_unique_types_3d',
    'txn_is_high_activity_3d',
    'txn_multi_channel_recent',
    'txn_amount_deviation_from_avg',
    'txn_night_txns_3d',
    'txn_weekend_txns_3d',
    'channel_new_jc_app',
    'channel_ussd',
    'channel_ussd_api',
    'channel_payment_gateway',
    'channel_mobile_app',
    'type_transfer_c2c',
    'type_transfer_c2b',
    'type_bill_payment',
    'type_mobile_load',
    'user_total_txns_3d',
    'user_total_amount_3d',
    'user_avg_amount_3d',
    'user_max_amount_3d',
    'user_unique_recipients_3d',
    'user_unique_channels_3d',
    'user_total_txns_7d',
    'user_avg_amount_7d',
    'user_max_amount_7d',
    'user_night_txns_7d',
    'user_weekend_txns_7d'
]

query = f"""
    SELECT {', '.join(selected_cols)}
    FROM stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
        AND mbar_account_type_name = 'Customer Account'
"""

subquery = f"({query}) AS fraud_data"

logger.info(f"🔍 Loading data from ClickHouse for date range: {start_date} to {end_date}")
start_time = datetime.now()

try:
    df = (spark.read
        .format('jdbc')
        .option('driver', driver)
        .option('url', url)
        .option('user', user)
        .option('password', password)
        .option('dbtable', subquery)
        .option('fetchsize', '100000')
        .option("partitionColumn", "cutoff_date")
        .option('lowerBound', start_date)
        .option('upperBound', end_date)
        .option('numPartitions', str(num_partitions))
        .load())
    
    # Cache for performance

    total_rows = df.count()
    
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    logger.info("✅ Data loaded successfully!")
    logger.info(f"   • Total rows: {total_rows:,}")
    logger.info(f"   • Load time: {duration:.2f} seconds")
    logger.info(f"   • Features: {len(selected_cols)}")
    
    # Check class distribution
    fraud_dist = df.groupBy('fraud_flag').count().orderBy('fraud_flag').collect()
    logger.info("📊 Fraud Flag Distribution:")
    for row in fraud_dist:
        pct = (row['count'] / total_rows) * 100
        logger.info(f"   • Class {row['fraud_flag']}: {row['count']:,} ({pct:.2f}%)")
    
    print(f"✅ Loaded {total_rows:,} transactions in {duration:.2f}s")
    df.show(5)
    
except Exception as e:
    logger.error(f"❌ Error loading data: {str(e)}")
    raise

2025-11-13 12:23:40,794 - INFO - 🔍 Loading data from ClickHouse for date range: 2025-06-01 to 2025-06-30
25/11/13 12:23:42 WARN JDBCRelation: The number of partitions is reduced because the specified number of partitions is less than the difference between upper bound and lower bound. Updated number of partitions: 29; Input number of partitions: 30; Lower bound: '2025-06-01'; Upper bound: '2025-06-30'.
25/11/13 12:23:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/11/13 12:24:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/11/13 12:24:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/11/13 12:24:43 WARN TaskSchedulerImpl: Initial job has not acce

KeyboardInterrupt: 

25/11/13 12:35:43 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/11/13 12:35:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/11/13 12:36:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/11/13 12:36:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/11/13 12:36:43 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
25/11/13 12:36:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure th

## 4. Data Preprocessing

In [ ]:
TARGET_COLUMN = 'fraud_flag'

# Columns to exclude from features
excluded_columns = [
    TARGET_COLUMN,
    'cutoff_date',
    'mbar_account_type_name'  # Already filtered to 'Customer Account'
]

# Get feature columns
all_feature_cols = [col_name for col_name in df.columns if col_name not in excluded_columns]

# Identify categorical (string) columns
string_cols = []
numeric_cols = []

for field in df.schema.fields:
    if field.name in all_feature_cols:
        if field.dataType.typeName() == 'string':
            string_cols.append(field.name)
        else:
            numeric_cols.append(field.name)

logger.info(f"🔤 Categorical features: {len(string_cols)}")
logger.info(f"   {string_cols}")
logger.info(f"🔢 Numeric features: {len(numeric_cols)}")

# Handle missing values in string columns
for col_name in string_cols:
    df = df.withColumn(col_name, 
                       when((col(col_name) == "") | col(col_name).isNull(), "UNKNOWN")
                       .otherwise(col(col_name)))

# Handle missing values in numeric columns (fill with 0)
for col_name in numeric_cols:
    df = df.withColumn(col_name, 
                       when(col(col_name).isNull(), 0.0)
                       .otherwise(col(col_name)))

logger.info("✅ Missing values handled")
print("✅ Data preprocessing complete!")

NameError: name 'df' is not defined

## 5. Feature Engineering for Decision Trees

In [ ]:
# String Indexing for categorical features (Decision Trees can handle indexed categories)
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep") 
    for col in string_cols
]

# Apply indexers
from pyspark.ml import Pipeline
indexer_pipeline = Pipeline(stages=indexers)
df_indexed = indexer_pipeline.fit(df).transform(df)

# Create feature list with indexed categorical columns
feature_cols = numeric_cols + [f"{col}_idx" for col in string_cols]

logger.info(f"🔧 Feature engineering complete:")
logger.info(f"   • Total features for modeling: {len(feature_cols)}")
logger.info(f"   • Numeric features: {len(numeric_cols)}")
logger.info(f"   • Indexed categorical features: {len(string_cols)}")

# Assemble features into a single vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

df_assembled = assembler.transform(df_indexed)
df_final = df_assembled.select("features", col(TARGET_COLUMN).alias("label"))

logger.info("✅ Feature vector assembled")
print(f"✅ Features assembled: {len(feature_cols)} features ready for Decision Tree")
df_final.show(5, truncate=False)

2025-11-12 16:47:51,088 - INFO - 🔧 Feature engineering complete:               
2025-11-12 16:47:51,089 - INFO -    • Total features for modeling: 48
2025-11-12 16:47:51,090 - INFO -    • Numeric features: 43
2025-11-12 16:47:51,091 - INFO -    • Indexed categorical features: 5
2025-11-12 16:47:51,088 - INFO - 🔧 Feature engineering complete:               
2025-11-12 16:47:51,089 - INFO -    • Total features for modeling: 48
2025-11-12 16:47:51,090 - INFO -    • Numeric features: 43
2025-11-12 16:47:51,091 - INFO -    • Indexed categorical features: 5
2025-11-12 16:47:51,159 - INFO - ✅ Feature vector assembled
2025-11-12 16:47:51,159 - INFO - ✅ Feature vector assembled


✅ Features assembled: 48 features ready for Decision Tree


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                                                                                                                       |label|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|(48,[0,1,2,3,4,

## 6. Split Data into Train/Test Sets

In [ ]:
# Split data: 80% training, 20% testing
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)


train_count = train_data.count()
test_count = test_data.count()

logger.info("📊 Data Split:")
logger.info(f"   • Training set: {train_count:,} samples ({train_count/(train_count+test_count)*100:.1f}%)")
logger.info(f"   • Test set: {test_count:,} samples ({test_count/(train_count+test_count)*100:.1f}%)")

print(f"✅ Data split complete: {train_count:,} train, {test_count:,} test")

2025-11-12 17:20:03,333 - INFO - Error while sending or receiving.
Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
BrokenPipeError: [Errno 32] Broken pipe
2025-11-12 17:20:03,335 - INFO - Closing down clientserver connection
2025-11-12 17:20:03,335 - INFO - Exception while sending command.
Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
BrokenPipeError: [Errno 32] Broken pipe

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/root/miniconda3/envs/fraud/lib/python3.10/si

KeyboardInterrupt: 

25/11/12 17:26:38 WARN TaskSetManager: Lost task 2.0 in stage 44.0 (TID 250) (10.205.161.118 executor 1): java.sql.SQLException: Failed to read value for column fraud_flag
	at com.clickhouse.jdbc.internal.ExceptionUtils.toSqlState(ExceptionUtils.java:69)
	at com.clickhouse.jdbc.internal.ExceptionUtils.toSqlState(ExceptionUtils.java:42)
	at com.clickhouse.jdbc.ResultSetImpl.next(ResultSetImpl.java:118)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.getNext(JdbcUtils.scala:372)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.getNext(JdbcUtils.scala:357)
	at org.apache.spark.util.NextIterator.hasNext(NextIterator.scala:73)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at org.apache.spark.util.CompletionIterator.hasNext(CompletionIterator.scala:31)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.sort_addToSorter_0$(Unknown Source)
	at org.apache.spark.sql.cataly

## 7. Train Decision Tree Classifier

In [ ]:
import time

# Configure Decision Tree with parameters optimized for rule extraction
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxDepth=10,  # Limit depth for interpretability
    maxBins=32,   # Bins for continuous features
    minInstancesPerNode=100,  # Prevent overfitting
    impurity="gini",  # Gini impurity for classification
    seed=42
)

logger.info("🌳 Decision Tree Configuration:")
logger.info(f"   • Max Depth: {dt.getMaxDepth()}")
logger.info(f"   • Max Bins: {dt.getMaxBins()}")
logger.info(f"   • Min Instances Per Node: {dt.getMinInstancesPerNode()}")
logger.info(f"   • Impurity Measure: {dt.getImpurity()}")

logger.info("🚀 Training Decision Tree model...")
start_time = time.time()

dt_model = dt.fit(train_data)

training_time = time.time() - start_time

logger.info("✅ Decision Tree trained successfully!")
logger.info(f"   • Training time: {training_time:.2f} seconds")
logger.info(f"   • Tree depth: {dt_model.depth}")
logger.info(f"   • Number of nodes: {dt_model.numNodes}")

print(f"✅ Decision Tree trained in {training_time:.2f}s")
print(f"   • Depth: {dt_model.depth}")
print(f"   • Nodes: {dt_model.numNodes}")

## 8. Model Evaluation

In [ ]:
# Make predictions on test set
predictions = dt_model.transform(test_data)

# Evaluate performance
binary_evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
multiclass_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

# Calculate metrics
auc_roc = binary_evaluator.evaluate(predictions)
accuracy = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "accuracy"})
precision = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "weightedPrecision"})
recall = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "weightedRecall"})
f1 = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "f1"})

logger.info("📊 Model Performance Metrics:")
logger.info(f"   • AUC-ROC: {auc_roc:.4f}")
logger.info(f"   • Accuracy: {accuracy:.4f}")
logger.info(f"   • Precision: {precision:.4f}")
logger.info(f"   • Recall: {recall:.4f}")
logger.info(f"   • F1-Score: {f1:.4f}")

# Confusion Matrix
confusion_matrix = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
logger.info("🔢 Confusion Matrix:")

print("\n📊 Model Performance:")
print(f"   • AUC-ROC: {auc_roc:.4f}")
print(f"   • Accuracy: {accuracy:.4f}")
print(f"   • Precision: {precision:.4f}")
print(f"   • Recall: {recall:.4f}")
print(f"   • F1-Score: {f1:.4f}")
print("\n🔢 Confusion Matrix:")
confusion_matrix.show()

## 9. Feature Importance Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Extract feature importance
feature_importance = dt_model.featureImportances.toArray()

# Create DataFrame with feature names and importance scores
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

# Log top features
logger.info("🎯 Top 20 Most Important Features:")
for idx, row in importance_df.head(20).iterrows():
    logger.info(f"   {row['feature']}: {row['importance']:.6f}")

# Display top 20 features
print("\n🎯 Top 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_n = 25
top_features = importance_df.head(top_n)

sns.barplot(data=top_features, y='feature', x='importance', palette='viridis')
plt.title(f'Top {top_n} Feature Importance - Decision Tree Fraud Detection', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()

# Save plot
importance_plot_path = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/dt_feature_importance.png'
plt.savefig(importance_plot_path, dpi=300, bbox_inches='tight')
logger.info(f"💾 Feature importance plot saved to: {importance_plot_path}")
plt.show()

# Save feature importance to CSV
importance_csv_path = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/dt_feature_importance.csv'
importance_df.to_csv(importance_csv_path, index=False)
logger.info(f"💾 Feature importance saved to: {importance_csv_path}")

print(f"\n✅ Feature importance analysis complete!")
print(f"   📊 Plot: {importance_plot_path}")
print(f"   📄 CSV: {importance_csv_path}")

## 10. Extract Decision Tree Rules

Convert the Decision Tree into human-readable if-then rules for fraud detection.

In [ ]:
def extract_rules_from_tree(tree_model, feature_names, class_names=['Legitimate', 'Fraud']):
    """
    Extract human-readable rules from a Decision Tree model.
    
    Args:
        tree_model: Trained DecisionTreeClassificationModel
        feature_names: List of feature names
        class_names: List of class labels
    
    Returns:
        List of rule dictionaries
    """
    tree_str = tree_model.toDebugString
    rules = []
    
    # Parse the debug string to extract rules
    lines = tree_str.split('\n')
    
    rule_count = 0
    for line in lines:
        if 'Predict:' in line:
            rule_count += 1
    
    logger.info(f"📋 Total decision rules in tree: {rule_count}")
    
    return tree_str, rule_count

# Extract rules
tree_rules, num_rules = extract_rules_from_tree(dt_model, feature_cols)

# Print the tree structure
print("\n🌳 Decision Tree Structure (First 50 lines):")
print("=" * 80)
print('\n'.join(tree_rules.split('\n')[:50]))
print("\n... (truncated) ...\n")

logger.info(f"✅ Extracted {num_rules} decision rules from tree")
print(f"\n✅ Tree has {num_rules} decision paths (rules)")

## 11. Extract Interpretable Fraud Rules

Parse the tree to extract actionable fraud detection rules.

In [ ]:
def parse_tree_rules(tree_str, feature_names, min_samples=100):
    """
    Parse decision tree into structured rules.
    
    Args:
        tree_str: Tree debug string
        feature_names: List of feature names
        min_samples: Minimum samples for a rule to be considered significant
    
    Returns:
        List of fraud rules
    """
    fraud_rules = []
    lines = tree_str.split('\n')
    
    current_path = []
    rule_id = 0
    
    for line in lines:
        # Check if this is a prediction line (leaf node)
        if 'Predict:' in line and '1.0' in line:  # Predict fraud (class 1)
            # Extract prediction info
            if 'Predict: 1.0' in line:
                rule_id += 1
                
                # Try to extract sample count
                # This is approximate as Spark doesn't provide exact counts in debug string
                rule = {
                    'rule_id': rule_id,
                    'prediction': 'FRAUD',
                    'path': ' AND '.join(current_path) if current_path else 'Root'
                }
                fraud_rules.append(rule)
        
        # Extract feature conditions
        if 'feature' in line.lower():
            # Parse feature index and threshold
            # This is a simplified parser - may need adjustment based on actual format
            pass
    
    return fraud_rules

# Extract fraud-specific rules
fraud_rules = parse_tree_rules(tree_rules, feature_cols)

logger.info(f"🎯 Extracted {len(fraud_rules)} fraud detection rules from tree")

# Display rules
print(f"\n🎯 Extracted {len(fraud_rules)} Fraud Detection Rules:")
print("=" * 80)
for rule in fraud_rules[:10]:  # Show first 10 rules
    print(f"\nRule {rule['rule_id']}: {rule['prediction']}")
    print(f"   Conditions: {rule['path']}")

if len(fraud_rules) > 10:
    print(f"\n... and {len(fraud_rules) - 10} more rules")

# Save full tree structure
tree_rules_path = '/root/research-dir/dev/jazzcash-fraud-detection/rules/decision_tree_full_structure.txt'
with open(tree_rules_path, 'w') as f:
    f.write(tree_rules)

logger.info(f"💾 Full tree structure saved to: {tree_rules_path}")
print(f"\n💾 Full tree structure saved to: {tree_rules_path}")

## 12. Train Random Forest for Comparison

Random Forest provides more robust feature importance through ensemble averaging.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Configure Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    numTrees=100,  # Number of trees in forest
    maxDepth=10,   # Max depth per tree
    maxBins=32,
    minInstancesPerNode=100,
    impurity="gini",
    subsamplingRate=0.8,  # Bootstrap sampling
    seed=42
)

logger.info("🌲 Random Forest Configuration:")
logger.info(f"   • Number of Trees: {rf.getNumTrees()}")
logger.info(f"   • Max Depth: {rf.getMaxDepth()}")
logger.info(f"   • Subsampling Rate: {rf.getSubsamplingRate()}")

logger.info("🚀 Training Random Forest...")
start_time = time.time()

rf_model = rf.fit(train_data)

rf_training_time = time.time() - start_time

logger.info("✅ Random Forest trained successfully!")
logger.info(f"   • Training time: {rf_training_time:.2f} seconds")
logger.info(f"   • Number of trees: {rf_model.getNumTrees}")

print(f"✅ Random Forest trained in {rf_training_time:.2f}s with {rf.getNumTrees()} trees")

## 13. Random Forest Feature Importance

In [ ]:
# Extract Random Forest feature importance
rf_feature_importance = rf_model.featureImportances.toArray()

# Create DataFrame
rf_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_feature_importance
}).sort_values('importance', ascending=False)

logger.info("🎯 Random Forest - Top 20 Most Important Features:")
for idx, row in rf_importance_df.head(20).iterrows():
    logger.info(f"   {row['feature']}: {row['importance']:.6f}")

print("\n🎯 Random Forest - Top 20 Most Important Features:")
print(rf_importance_df.head(20).to_string(index=False))

# Compare Decision Tree vs Random Forest
comparison_df = importance_df.merge(
    rf_importance_df, 
    on='feature', 
    suffixes=('_dt', '_rf')
).head(25)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Decision Tree
sns.barplot(data=importance_df.head(20), y='feature', x='importance', ax=axes[0], palette='Blues_r')
axes[0].set_title('Decision Tree Feature Importance', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance Score', fontsize=12)
axes[0].set_ylabel('Feature', fontsize=12)

# Random Forest
sns.barplot(data=rf_importance_df.head(20), y='feature', x='importance', ax=axes[1], palette='Greens_r')
axes[1].set_title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Importance Score', fontsize=12)
axes[1].set_ylabel('Feature', fontsize=12)

plt.tight_layout()

# Save comparison plot
comparison_plot_path = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/dt_rf_feature_importance_comparison.png'
plt.savefig(comparison_plot_path, dpi=300, bbox_inches='tight')
logger.info(f"💾 Comparison plot saved to: {comparison_plot_path}")
plt.show()

# Save Random Forest importance
rf_importance_csv_path = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/rf_feature_importance.csv'
rf_importance_df.to_csv(rf_importance_csv_path, index=False)
logger.info(f"💾 Random Forest importance saved to: {rf_importance_csv_path}")

print(f"\n✅ Feature importance comparison complete!")
print(f"   📊 Comparison plot: {comparison_plot_path}")
print(f"   📄 RF CSV: {rf_importance_csv_path}")

## 14. Evaluate Random Forest Performance

In [ ]:
# Make predictions with Random Forest
rf_predictions = rf_model.transform(test_data)

# Evaluate Random Forest
rf_auc_roc = binary_evaluator.evaluate(rf_predictions)
rf_accuracy = multiclass_evaluator.evaluate(rf_predictions, {multiclass_evaluator.metricName: "accuracy"})
rf_precision = multiclass_evaluator.evaluate(rf_predictions, {multiclass_evaluator.metricName: "weightedPrecision"})
rf_recall = multiclass_evaluator.evaluate(rf_predictions, {multiclass_evaluator.metricName: "weightedRecall"})
rf_f1 = multiclass_evaluator.evaluate(rf_predictions, {multiclass_evaluator.metricName: "f1"})

logger.info("📊 Random Forest Performance:")
logger.info(f"   • AUC-ROC: {rf_auc_roc:.4f}")
logger.info(f"   • Accuracy: {rf_accuracy:.4f}")
logger.info(f"   • Precision: {rf_precision:.4f}")
logger.info(f"   • Recall: {rf_recall:.4f}")
logger.info(f"   • F1-Score: {rf_f1:.4f}")

# Performance comparison
print("\n📊 Model Performance Comparison:")
print("=" * 60)
print(f"{'Metric':<20} {'Decision Tree':<20} {'Random Forest':<20}")
print("=" * 60)
print(f"{'AUC-ROC':<20} {auc_roc:<20.4f} {rf_auc_roc:<20.4f}")
print(f"{'Accuracy':<20} {accuracy:<20.4f} {rf_accuracy:<20.4f}")
print(f"{'Precision':<20} {precision:<20.4f} {rf_precision:<20.4f}")
print(f"{'Recall':<20} {recall:<20.4f} {rf_recall:<20.4f}")
print(f"{'F1-Score':<20} {f1:<20.4f} {rf_f1:<20.4f}")
print("=" * 60)

# Save performance comparison
performance_comparison = pd.DataFrame({
    'Metric': ['AUC-ROC', 'Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Decision_Tree': [auc_roc, accuracy, precision, recall, f1],
    'Random_Forest': [rf_auc_roc, rf_accuracy, rf_precision, rf_recall, rf_f1]
})

performance_csv_path = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/dt_rf_performance_comparison.csv'
performance_comparison.to_csv(performance_csv_path, index=False)
logger.info(f"💾 Performance comparison saved to: {performance_csv_path}")
print(f"\n💾 Performance comparison saved to: {performance_csv_path}")

## 15. Save Models

In [ ]:
# Save Decision Tree model
dt_model_path = "/root/research-dir/dev/jazzcash-fraud-detection/models/decision_tree_fraud_model"
dt_model.write().overwrite().save(dt_model_path)
logger.info(f"💾 Decision Tree model saved to: {dt_model_path}")

# Save Random Forest model
rf_model_path = "/root/research-dir/dev/jazzcash-fraud-detection/models/random_forest_fraud_model"
rf_model.write().overwrite().save(rf_model_path)
logger.info(f"💾 Random Forest model saved to: {rf_model_path}")

# Save feature names and metadata
metadata = {
    "training_date": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "date_range": f"{start_date} to {end_date}",
    "total_features": len(feature_cols),
    "numeric_features": len(numeric_cols),
    "categorical_features": len(string_cols),
    "feature_names": feature_cols,
    "string_columns": string_cols,
    "numeric_columns": numeric_cols,
    "dt_performance": {
        "auc_roc": float(auc_roc),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "tree_depth": dt_model.depth,
        "num_nodes": dt_model.numNodes
    },
    "rf_performance": {
        "auc_roc": float(rf_auc_roc),
        "accuracy": float(rf_accuracy),
        "precision": float(rf_precision),
        "recall": float(rf_recall),
        "f1_score": float(rf_f1),
        "num_trees": rf.getNumTrees()
    }
}

metadata_path = "/root/research-dir/dev/jazzcash-fraud-detection/models/tree_models_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

logger.info(f"💾 Metadata saved to: {metadata_path}")

print("\n✅ Models and metadata saved successfully!")
print(f"   🌳 Decision Tree: {dt_model_path}")
print(f"   🌲 Random Forest: {rf_model_path}")
print(f"   📄 Metadata: {metadata_path}")

## 16. Generate Actionable Fraud Detection Rules

Create SQL-like rules based on top feature importance.

In [ ]:
# Generate human-readable rules from top features
top_features = importance_df.head(10)

print("\n🎯 ACTIONABLE FRAUD DETECTION RULES")
print("=" * 80)
print("Based on Decision Tree Feature Importance Analysis\n")

rule_templates = []

for idx, row in top_features.iterrows():
    feature_name = row['feature']
    importance = row['importance']
    
    print(f"\nRule {idx + 1}: {feature_name}")
    print(f"   Importance: {importance:.4f}")
    
    # Generate rule suggestions based on feature type
    if 'txn_txns' in feature_name:
        print(f"   💡 Rule: Flag if transaction count in time window is abnormally high")
        print(f"   📝 SQL: WHERE {feature_name} > threshold_value")
    elif 'amount' in feature_name:
        print(f"   💡 Rule: Flag if transaction amount deviates significantly from user's pattern")
        print(f"   📝 SQL: WHERE {feature_name} > user_avg * 3")
    elif 'night' in feature_name or 'unusual' in feature_name:
        print(f"   💡 Rule: Flag if transaction occurs during unusual hours")
        print(f"   📝 SQL: WHERE {feature_name} = 1")
    elif 'channel' in feature_name:
        print(f"   💡 Rule: Flag if multiple channels used in short time or suspicious channel")
        print(f"   📝 SQL: WHERE {feature_name} = suspicious_value")
    elif 'recipient' in feature_name:
        print(f"   💡 Rule: Flag if sending to unusual number of recipients")
        print(f"   📝 SQL: WHERE {feature_name} > normal_threshold")
    else:
        print(f"   💡 Rule: Monitor this feature for anomalous values")
        print(f"   📝 SQL: WHERE {feature_name} outside normal_range")

print("\n" + "=" * 80)
print("✅ Rules generated! Use these insights to create production fraud detection rules.")
print("💡 Tip: Combine multiple rules using AND/OR logic for better precision.")

## 17. Summary Report

In [ ]:
print("\n" + "=" * 80)
print("📊 DECISION TREE FRAUD DETECTION - SUMMARY REPORT")
print("=" * 80)

print(f"\n📅 Training Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Dataset: {start_date} to {end_date}")
print(f"🔢 Total Samples: {train_count + test_count:,}")
print(f"   • Training: {train_count:,}")
print(f"   • Testing: {test_count:,}")

print(f"\n🎯 Features:")
print(f"   • Total Features: {len(feature_cols)}")
print(f"   • Numeric: {len(numeric_cols)}")
print(f"   • Categorical: {len(string_cols)}")

print(f"\n🌳 Decision Tree Model:")
print(f"   • Depth: {dt_model.depth}")
print(f"   • Nodes: {dt_model.numNodes}")
print(f"   • AUC-ROC: {auc_roc:.4f}")
print(f"   • Accuracy: {accuracy:.4f}")
print(f"   • Precision: {precision:.4f}")
print(f"   • Recall: {recall:.4f}")
print(f"   • F1-Score: {f1:.4f}")

print(f"\n🌲 Random Forest Model:")
print(f"   • Trees: {rf.getNumTrees()}")
print(f"   • AUC-ROC: {rf_auc_roc:.4f}")
print(f"   • Accuracy: {rf_accuracy:.4f}")
print(f"   • Precision: {rf_precision:.4f}")
print(f"   • Recall: {rf_recall:.4f}")
print(f"   • F1-Score: {rf_f1:.4f}")

print(f"\n🎯 Top 5 Most Important Features:")
for idx, row in importance_df.head(5).iterrows():
    print(f"   {idx + 1}. {row['feature']}: {row['importance']:.4f}")

print(f"\n💾 Saved Artifacts:")
print(f"   • Decision Tree Model: {dt_model_path}")
print(f"   • Random Forest Model: {rf_model_path}")
print(f"   • Feature Importance (DT): {importance_csv_path}")
print(f"   • Feature Importance (RF): {rf_importance_csv_path}")
print(f"   • Tree Structure: {tree_rules_path}")
print(f"   • Performance Comparison: {performance_csv_path}")
print(f"   • Metadata: {metadata_path}")
print(f"   • Log File: {log_path}")

print("\n" + "=" * 80)
print("✅ Decision Tree analysis complete!")
print("💡 Next steps:")
print("   1. Review extracted rules and feature importance")
print("   2. Convert high-importance features into production rules")
print("   3. Test rules on validation dataset")
print("   4. Integrate rules into fraud detection system")
print("=" * 80)

logger.info("✅ Decision Tree Fraud Detection Analysis Complete!")